# Exceedance plots from a SimCLR checkpoint

This is a cleaned version of `analyze_exceedances.ipynb`. It keeps only the checkpoint loading, score collection, POT fitting, and the first four exceedance plots.

Paste a Google Drive checkpoint link or local checkpoint path in the configuration cell below. Everything after the original shortfall testing section has been removed.

In [ ]:
# Uncomment if running in a fresh environment.
# %pip install -q gdown scipy pyyaml tqdm


In [ ]:
from pathlib import Path
import os
import re
import sys
import math
import random
from types import SimpleNamespace
from typing import Any, Dict, Optional

import numpy as np
import torch
import torch.nn.functional as F
from torch.utils.data import DataLoader
import torchvision
from torchvision.datasets import ImageFolder
import matplotlib.pyplot as plt
from tqdm.auto import tqdm
import yaml

# ---------------------------------------------------------------------
# User configuration
# ---------------------------------------------------------------------

# Paste either a local checkpoint path or a Google Drive file link.
# Example: "https://drive.google.com/file/d/<FILE_ID>/view?usp=sharing"
CHECKPOINT_SOURCE = "PASTE_GOOGLE_DRIVE_LINK_OR_LOCAL_CHECKPOINT_PATH_HERE"

# Used only when CHECKPOINT_SOURCE is a URL.
CHECKPOINT_PATH = "./checkpoints/exceedance_checkpoint.tar"

# If config.yaml is next to a local checkpoint, these values are overridden.
DATASET = "CIFAR100"      # CIFAR10, CIFAR100, STL10, IMAGENET32, IMAGENET
DATA_ROOT = "./data"
RESNET = "resnet18"
PROJECTION_DIM = 64
IMAGE_SIZE = 32
BATCH_SIZE = 256
REPEATS = 64
SEED = 42

# POT threshold quantile. The original notebook used q=0.99.
POT_Q = 0.99
MIN_EXCEEDANCES = 10

FIG_DIR = Path("notebooks/figs")
FIG_DIR.mkdir(parents=True, exist_ok=True)


In [ ]:
# ---------------------------------------------------------------------
# Repository and checkpoint loading helpers
# ---------------------------------------------------------------------

def find_repo_root() -> Path:
    cwd = Path.cwd().resolve()
    for candidate in [cwd, cwd.parent, cwd.parent.parent]:
        if (candidate / "simclr").exists() and (candidate / "model.py").exists():
            return candidate
    raise FileNotFoundError(
        "Could not locate the weibitSimCLR repo root. Run from the repo root or from repo_root/notebooks."
    )

REPO_ROOT = find_repo_root()
os.chdir(REPO_ROOT)
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))
print("Repo root:", REPO_ROOT)

from simclr import SimCLR
from simclr.modules import get_resnet_adaptable
from simclr.modules.transformations import TransformsSimCLR


def set_seed(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


def google_drive_file_id(url: str) -> Optional[str]:
    patterns = [r"/file/d/([a-zA-Z0-9_-]+)", r"[?&]id=([a-zA-Z0-9_-]+)"]
    for pat in patterns:
        m = re.search(pat, url)
        if m:
            return m.group(1)
    return None


def resolve_checkpoint(source: str, out_path: str) -> Path:
    source = str(source).strip()
    if not source or source.startswith("PASTE_"):
        raise ValueError("Set CHECKPOINT_SOURCE to a checkpoint path or Google Drive file link.")

    local = Path(source).expanduser()
    if local.exists():
        return local.resolve()

    out = Path(out_path).expanduser().resolve()
    out.parent.mkdir(parents=True, exist_ok=True)

    if "drive.google.com" in source:
        file_id = google_drive_file_id(source)
        if file_id is None:
            raise ValueError("Could not parse a Google Drive file id from CHECKPOINT_SOURCE.")
        try:
            import gdown
        except ImportError as e:
            raise ImportError("Install gdown first, e.g. `%pip install gdown`.") from e
        url = f"https://drive.google.com/uc?id={file_id}"
        print(f"Downloading checkpoint to {out}")
        gdown.download(url, str(out), quiet=False)
        return out

    if source.startswith("http://") or source.startswith("https://"):
        import urllib.request
        print(f"Downloading checkpoint to {out}")
        urllib.request.urlretrieve(source, out)
        return out

    raise FileNotFoundError(f"Unsupported checkpoint source: {source}")


def load_nearby_config(checkpoint_path: Path) -> Dict[str, Any]:
    for d in [checkpoint_path.parent, checkpoint_path.parent.parent]:
        cfg = d / "config.yaml"
        if cfg.exists():
            print("Loaded config:", cfg)
            with cfg.open("r") as f:
                return yaml.safe_load(f) or {}
    print("No config.yaml found next to checkpoint. Using notebook parameters.")
    return {}


def strip_module_prefix(sd: Dict[str, torch.Tensor]) -> Dict[str, torch.Tensor]:
    if sd and all(k.startswith("module.") for k in sd.keys()):
        return {k.replace("module.", "", 1): v for k, v in sd.items()}
    return sd


def extract_state_dict(payload: Any) -> Dict[str, torch.Tensor]:
    if isinstance(payload, dict):
        for key in ["state_dict", "model_state_dict", "model", "model_state"]:
            if key in payload and isinstance(payload[key], dict):
                return strip_module_prefix(payload[key])
        tensor_items = {k: v for k, v in payload.items() if isinstance(v, torch.Tensor)}
        if tensor_items:
            return strip_module_prefix(tensor_items)
    raise ValueError("Could not find a model state_dict in the checkpoint.")


In [ ]:
# ---------------------------------------------------------------------
# Resolve checkpoint, dataset, and model
# ---------------------------------------------------------------------
set_seed(SEED)
checkpoint_path = resolve_checkpoint(CHECKPOINT_SOURCE, CHECKPOINT_PATH)
config = load_nearby_config(checkpoint_path)

DATASET = str(config.get("dataset", DATASET)).upper()
DATA_ROOT = str(config.get("dataset_dir", DATA_ROOT))
RESNET = str(config.get("resnet", RESNET))
PROJECTION_DIM = int(config.get("projection_dim", PROJECTION_DIM))
IMAGE_SIZE = int(config.get("image_size", IMAGE_SIZE))
BATCH_SIZE = int(config.get("batch_size", BATCH_SIZE))

args = SimpleNamespace(
    dataset=DATASET,
    dataset_dir=DATA_ROOT,
    resnet=RESNET,
    projection_dim=PROJECTION_DIM,
    image_size=IMAGE_SIZE,
    batch_size=BATCH_SIZE,
)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

if args.dataset == "STL10":
    train_dataset = torchvision.datasets.STL10(
        args.dataset_dir, split="unlabeled", download=True,
        transform=TransformsSimCLR(size=args.image_size),
    )
elif args.dataset == "CIFAR10":
    train_dataset = torchvision.datasets.CIFAR10(
        args.dataset_dir, download=True, train=True,
        transform=TransformsSimCLR(size=args.image_size),
    )
elif args.dataset == "CIFAR100":
    train_dataset = torchvision.datasets.CIFAR100(
        args.dataset_dir, train=True, download=True,
        transform=TransformsSimCLR(size=args.image_size),
    )
elif args.dataset == "IMAGENET32":
    from utils.imagenet32 import ImageNet32
    train_dataset = ImageNet32(
        root=args.dataset_dir, train=True,
        transform=TransformsSimCLR(size=args.image_size),
    )
elif args.dataset == "IMAGENET":
    train_dataset = ImageFolder(
        os.path.join(args.dataset_dir, "train"),
        transform=TransformsSimCLR(size=args.image_size),
    )
else:
    raise NotImplementedError(args.dataset)

encoder = get_resnet_adaptable(args.resnet, args.dataset, pretrained=False)
n_features = encoder.fc.in_features
model = SimCLR(encoder, args.projection_dim, n_features=n_features)

payload = torch.load(checkpoint_path, map_location="cpu")
state_dict = extract_state_dict(payload)
missing, unexpected = model.load_state_dict(state_dict, strict=False)
if missing:
    print("Missing keys:", missing[:10], "..." if len(missing) > 10 else "")
if unexpected:
    print("Unexpected keys:", unexpected[:10], "..." if len(unexpected) > 10 else "")

model = model.to(device).eval()
print("Checkpoint:", checkpoint_path)
print("Dataset:", args.dataset)
print("Backbone:", args.resnet)
print("Batch size:", args.batch_size)
print("Device:", device)


## Testing the shortfall assumption

The next cells collect a representative aggregate tail of negative cosine similarities from the frozen checkpoint, fit a POT GPD above a high threshold, and produce the four exceedance plots. The notebook intentionally stops after these plots.

In [ ]:
# ---------------------------------------------------------------------
# Score collection and POT fitting
# ---------------------------------------------------------------------

@torch.no_grad()
def collect_neg_scores(model, dataset, K, device, repeats=16):
    # This follows the original notebook: sample batches of K+1 augmented views,
    # compute z1 @ z1.T, and keep the lower-triangular off-diagonal entries.
    model.eval()
    negs = []
    loader = DataLoader(
        dataset,
        batch_size=K + 1,
        shuffle=True,
        drop_last=True,
        num_workers=0,
        pin_memory=False,
    )
    it = iter(loader)
    tri_mask = torch.triu(torch.ones(K + 1, K + 1, device=device), diagonal=0).bool()

    for _ in tqdm(range(repeats), desc="Collecting negative scores"):
        try:
            (x1, x2), _ = next(it)
        except StopIteration:
            it = iter(loader)
            (x1, x2), _ = next(it)

        x1 = x1.to(device, non_blocking=True)
        x2 = x2.to(device, non_blocking=True)
        h1, h2, z1, z2 = model(x1, x2)
        z1 = F.normalize(z1, p=2, dim=1)
        sim = z1 @ z1.T
        sim[tri_mask] = -float("inf")
        negs.append(sim.detach().cpu().numpy())

    return np.concatenate(negs, axis=0)


def pot_gpd_fit(scores,
                q: float = 0.99,
                u: float = None,
                min_exceedances: int = 200,
                xi_bounds=(-1.5, 1.0),
                xi_grid_size: int = 400,
                refine: bool = True,
                verbose: bool = False):
    # Cap-free POT GPD MLE on the upper tail of scores.
    s = np.asarray(scores, dtype=np.float64)
    s = s[np.isfinite(s)]
    if s.size == 0:
        raise ValueError("No finite scores.")

    if u is None:
        u = np.quantile(s, q)
    y = s[s > u] - u
    n = y.size
    if n < min_exceedances:
        raise ValueError(f"Not enough exceedances above u={u:.6g}: n={n} < {min_exceedances}.")

    y.sort()
    y_max = y[-1]
    y_bar = y.mean()

    def loglik(xi, sigma):
        if not np.isfinite(xi) or not np.isfinite(sigma) or sigma <= 0:
            return -np.inf
        if abs(xi) < 1e-8:
            return -n * np.log(sigma) - (y / sigma).sum()
        t = 1.0 + xi * (y / sigma)
        if np.any(t <= 0.0):
            return -np.inf
        return -n * np.log(sigma) - (1.0 + 1.0 / xi) * np.log(t).sum()

    def sigma_mle_given_xi(xi):
        if abs(xi) < 1e-8:
            return max(y_bar, 1e-12)
        sig_lo = (abs(xi) * y_max + 1e-12) if (xi < 0) else 1e-12
        sigma = max(sig_lo * 1.05, y_bar)
        for _ in range(100):
            t = 1.0 + xi * (y / sigma)
            if np.any(t <= 0.0) or not np.isfinite(t).all():
                sigma = max(sigma * 1.5, sig_lo * 1.1)
                continue
            denom = sigma + xi * y
            g = (n / (xi * sigma)) - (1.0 + 1.0 / xi) * (1.0 / denom).sum()
            gp = (-n / (xi * sigma ** 2)) + (1.0 + 1.0 / xi) * (1.0 / denom ** 2).sum()
            if not np.isfinite(g) or not np.isfinite(gp) or gp == 0:
                break
            step = g / gp
            sigma_new = sigma - step
            if sigma_new <= sig_lo or not np.isfinite(sigma_new):
                sigma_new = 0.5 * (sigma + sig_lo * 1.01)
            if abs(sigma_new - sigma) / max(sigma, 1e-12) < 1e-10:
                sigma = sigma_new
                break
            sigma = sigma_new
        return max(sigma, sig_lo * 1.000001)

    xis = np.linspace(xi_bounds[0], xi_bounds[1], xi_grid_size)
    best = (-np.inf, None, None)
    for xi in xis:
        sigma = sigma_mle_given_xi(float(xi))
        ll = loglik(float(xi), sigma)
        if ll > best[0]:
            best = (ll, float(xi), float(sigma))

    if refine:
        xi0 = best[1]
        width = (xi_bounds[1] - xi_bounds[0]) / xi_grid_size * 4
        lo = max(xi_bounds[0], xi0 - width)
        hi = min(xi_bounds[1], xi0 + width)
        xis_ref = np.linspace(lo, hi, xi_grid_size)
        for xi in xis_ref:
            sigma = sigma_mle_given_xi(float(xi))
            ll = loglik(float(xi), sigma)
            if ll > best[0]:
                best = (ll, float(xi), float(sigma))

    _, xi_hat, sigma_hat = best
    xF_hat = u - sigma_hat / xi_hat if xi_hat < 0 else np.inf
    if verbose:
        print(f"n={s.size}, u={u:.6g}, exceedances={n}")
        print(f"xi_hat={xi_hat:.6g}, sigma_hat={sigma_hat:.6g}, xF_hat={xF_hat:.6g}")
    return xi_hat, sigma_hat, u, xF_hat

scores = collect_neg_scores(model, train_dataset, args.batch_size, device, repeats=REPEATS)
scores_ = scores[np.isfinite(scores)]
xi_hat, sigma_hat, u, xF_hat = pot_gpd_fit(scores_, q=POT_Q, min_exceedances=MIN_EXCEEDANCES, refine=True, verbose=True)


In [ ]:
# ---------------------------------------------------------------------
# Plot helpers
# ---------------------------------------------------------------------

def gpd_pdf_cdf(y, xi, sigma):
    y = np.asarray(y, dtype=np.float64)
    if abs(xi) < 1e-10:
        pdf = (1.0 / sigma) * np.exp(-y / sigma)
        cdf = 1.0 - np.exp(-y / sigma)
        support_max = np.inf
    else:
        t = 1.0 + xi * (y / sigma)
        pdf = np.zeros_like(y)
        cdf = np.zeros_like(y)
        valid = t > 0
        pdf[valid] = (1.0 / sigma) * np.power(t[valid], -(1.0 / xi + 1.0))
        cdf[valid] = 1.0 - np.power(t[valid], -1.0 / xi)
        support_max = (-sigma / xi) if xi < 0 else np.inf
    return pdf, cdf, support_max


def plot_pot_fit_gumbel_vs_weibull(scores, xi, sigma, u, bins=60, max_points=50000,
                                   figsize=(7, 4.2), loglog=False, annotate=False):
    s = np.asarray(scores, dtype=np.float64)
    s = s[np.isfinite(s)]
    y = s[s > u] - u
    if y.size == 0:
        raise ValueError("No exceedances above the threshold u.")
    if y.size > max_points:
        rng = np.random.default_rng(SEED)
        y = rng.choice(y, size=max_points, replace=False)

    _, _, supp_max = gpd_pdf_cdf([0.0], xi, sigma)
    y_max = y.max()
    grid_max = min(y_max * 1.05, supp_max * 0.999 if np.isfinite(supp_max) else y_max * 1.05)
    grid = np.linspace(0.0, grid_max, 400)
    pdf, _, _ = gpd_pdf_cdf(grid, xi, sigma)
    pdf_exp, _, _ = gpd_pdf_cdf(grid, 0.0, sigma)

    plt.figure(figsize=figsize)
    plt.hist(y, bins=bins, density=True, alpha=0.2, edgecolor="black", linewidth=1,
             label="exceedance histogram")
    plt.plot(grid, pdf, "k--", lw=2, label=f"Weibull-GPD(xi={xi:.3f}, sigma={sigma:.3f})")
    plt.plot(grid, pdf_exp, "r--", label="Gumbel-GPD(xi=0)")
    if np.isfinite(supp_max):
        plt.axvline(supp_max, color="g", ls="-", lw=2, label="cap")
    if annotate:
        plt.title("POT exceedances")
    plt.legend(loc="upper center", ncol=4, bbox_to_anchor=(0.5, -0.2), facecolor="lightblue", framealpha=1)
    plt.xlabel(r"exceedances ($y = s-u$)")
    plt.ylabel("density")
    if np.isfinite(supp_max):
        plt.xlim(max(supp_max * 0.1, 1e-12), supp_max * 1.05)
    if loglog:
        plt.xscale("log")
        plt.yscale("log")
        plt.ylim(1e-4, 1e2)
    else:
        plt.ylim(0, 1.2 * np.nanmax(pdf))
    plt.tight_layout()
    path = FIG_DIR / ("pot_fit_loglog.pdf" if loglog else "pot_fit.pdf")
    plt.savefig(path)
    print("saved", path)
    plt.show()


def plot_qq_fit(scores, xi, sigma, u, max_points=50000, figsize=(7, 4.2), annotate=False):
    s = np.asarray(scores, dtype=np.float64)
    s = s[np.isfinite(s)]
    y = s[s > u] - u
    if y.size == 0:
        raise ValueError("No exceedances above the threshold u.")
    if y.size > max_points:
        rng = np.random.default_rng(SEED)
        y = rng.choice(y, size=max_points, replace=False)

    probs = np.linspace(0.01, 0.99, 99)
    if abs(xi) < 1e-10:
        q_fit = sigma * (-np.log(1 - probs))
    else:
        q_fit = (sigma / xi) * ((1 - probs) ** (-xi) - 1.0)
        if xi < 0:
            q_fit = np.clip(q_fit, 0, -sigma / xi * 0.999)
    q_emp = np.quantile(y, probs)
    q_fit_gumbel = sigma * (-np.log(1 - probs))

    lim = [0, max(q_fit.max(), q_emp.max(), q_fit_gumbel.max()) * 1.05]
    plt.figure(figsize=figsize)
    plt.plot(q_fit, q_emp, "--o", lw=2, ms=3, color="tab:blue", label=f"Weibull-GPD(xi={xi:.3f}, sigma={sigma:.3f})")
    plt.plot(q_fit_gumbel, q_emp, "--x", lw=2, ms=3, color="tab:red", label="Gumbel-GPD(xi=0)")
    plt.plot(lim, lim, "g-", lw=1, label="y=x", alpha=0.5)
    plt.xlabel("fitted GPD quantiles")
    plt.ylabel("empirical quantiles")
    plt.xlim(lim)
    plt.ylim(lim)
    plt.grid(True)
    if annotate:
        plt.title("QQ: GPD quantiles vs empirical")
    plt.legend(loc="upper center", ncol=4, bbox_to_anchor=(0.5, -0.2), facecolor="lightblue", framealpha=1)
    plt.tight_layout()
    path = FIG_DIR / "pot_fit_qq.pdf"
    plt.savefig(path)
    print("saved", path)
    plt.show()


def pot_shape_stability(scores, q_list=(0.98, 0.985, 0.99, 0.992, 0.995),
                        min_exceedances=100, annotate=False, figsize=(7, 4.2)):
    xis, us = [], []
    for q in tqdm(q_list, desc="POT thresholds"):
        try:
            xi, sigma, u_q, _ = pot_gpd_fit(scores, q=q, min_exceedances=min_exceedances, refine=True, verbose=False)
            xis.append(xi)
            us.append(u_q)
        except ValueError:
            pass
    plt.figure(figsize=figsize)
    plt.plot(us, xis, "o-", lw=1.5)
    plt.axhline(0.0, color="k", ls="--", lw=1)
    plt.xlabel("threshold u")
    plt.ylabel("xi (GPD shape)")
    if annotate:
        plt.title("POT shape stability")
    plt.grid(True)
    plt.tight_layout()
    path = FIG_DIR / "pot_shape_stability.pdf"
    plt.savefig(path)
    print("saved", path)
    plt.show()


In [ ]:
# Plot 1: log-log POT exceedance density
plot_pot_fit_gumbel_vs_weibull(
    scores_, xi_hat, sigma_hat, u,
    bins=40, max_points=50000, figsize=(7, 4.2), loglog=True, annotate=False,
)


In [ ]:
# Plot 2: linear-scale POT exceedance density
plot_pot_fit_gumbel_vs_weibull(
    scores_, xi_hat, sigma_hat, u,
    bins=40, max_points=50000, figsize=(7, 4.2), loglog=False, annotate=False,
)


In [ ]:
# Plot 3: QQ plot for Weibull-GPD and Gumbel-GPD fits
plot_qq_fit(scores_, xi_hat, sigma_hat, u, max_points=50000, figsize=(7, 4.2), annotate=False)


In [ ]:
# Plot 4: POT shape stability across nearby high thresholds
pot_shape_stability(scores_, annotate=False, figsize=(7, 4.2))


## End

This cleaned notebook stops here. The later shortfall-collapse tests, likelihood-ratio tests, and other exploratory analyses from the original notebook have been removed.